# Notebook 03 — Claude-as-Judge Evaluation (Figure 1)

**Paper:** *Graph-Structured Reinforcement Learning for Scientific Hypothesis Generation in Materials Design*

This notebook implements the **reasoning trace evaluation** pipeline described in **Section 2.1**, producing the bar-chart scores shown in **Figure 1**.

**Judge model:** Claude `claude-opus-4-7` (used instead of OpenAI GPT to avoid model-family bias; Section 2.1)

**Models evaluated (6 total):**

| Model | Scale | Base |
|---|---|---|
| Graph-PRefLexOR-8B  | 8B   | Qwen3-8B |
| Graph-PRefLexOR-3B  | 3B   | Llama-3.2-3B-Instruct |
| Graph-PRefLexOR-1.7B | 1.7B | Qwen3-1.7B |
| Qwen3-8B (thinking) | 8B   | — |
| Qwen3-8B (no-think) | 8B   | — |
| Llama-3.2-3B-Instruct | 3B  | — |
| Qwen3-1.7B (thinking) | 1.7B | — |
| Qwen3-1.7B (no-think) | 1.7B | — |

**Metrics (0–10 scale):**
- **Reasoning Quality** — mechanistic correctness and causal grounding
- **Intellectual Depth** — cross-scale synthesis and novelty
- **Reasoning Traceability** — structural organization and causal transparency
- **Overall Score** — mean of the three above

**Input data:** `data/results/*_data_eval_all.jsonl` (model inference outputs)  
**Output data:** `data/results/*_res_eval.jsonl` (evaluation scores per response)  
**Prerequisites:** `pip install anthropic pandas matplotlib pydantic python-dotenv`

## 1. Setup and Configuration

In [ ]:
import os
import re
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import anthropic
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pathlib import Path
from tqdm import tqdm

# Load Anthropic API key from repo root .env
load_dotenv(Path("../.env"))
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# Paths relative to repo root (one level up from notebooks/)
DATA_DIR    = Path("../data/results")
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)


## 2. Evaluation Schema

Three criteria are scored 0–10. `overall_score` is their mean.

In [ ]:
class CriterionScore(BaseModel):
    score:         float = Field(ge=0, le=10)
    justification: str

class EvalResult(BaseModel):
    reasoning_quality:      CriterionScore   # mechanistic correctness
    intellectual_depth:     CriterionScore   # cross-scale synthesis
    reasoning_traceability: CriterionScore   # structural organization
    overall_score:          float            # mean of the three

    model_config = {"arbitrary_types_allowed": True}


## 3. Evaluation Prompt

The judge receives the question, the ground-truth answer, and the model's **reasoning trace** (the `<think>` block). For Llama-3.2-3B-Instruct, which has no `<think>` phase, the final answer is evaluated directly.

In [ ]:
EVAL_SYSTEM = """You are a rigorous scientific peer reviewer evaluating LLM responses on PhD journal-club level open-ended questions.

SCORING CRITERIA (0–10 each):
1. Reasoning Quality      — Are mechanisms correct, causally grounded, and domain-appropriate?
2. Intellectual Depth     — Does the response synthesize across scales, identify non-obvious connections?
3. Reasoning Traceability — Is the reasoning transparent, structured, and easy to follow causally?

overall_score = mean(reasoning_quality, intellectual_depth, reasoning_traceability)

Provide a brief justification (1–2 sentences) for each score.
"""

EVAL_USER_TEMPLATE = """Model: {model_name}

QUESTION:
{question}

GROUND-TRUTH ANSWER:
{correct_answer}

MODEL REASONING TRACE (or final answer if no trace):
{model_reasoning}

Score each criterion 0–10. Return a JSON object with keys:
reasoning_quality, intellectual_depth, reasoning_traceability, overall_score.
Each criterion value: {{"score": <float>, "justification": "<text>"}}.
"""

def evaluate(model_name: str, question: str, correct_answer: str,
             model_reasoning: str = "") -> EvalResult:
    """Score a single model response using Claude as judge."""
    user_msg = EVAL_USER_TEMPLATE.format(
        model_name      = model_name,
        question        = question,
        correct_answer  = correct_answer,
        model_reasoning = model_reasoning or "EMPTY",
    )
    # Use claude-opus-4-7 as judge (avoids OpenAI model-family bias per Section 2.1)
    resp = client.messages.create(
        model      = "claude-opus-4-7",
        max_tokens = 1024,
        system     = EVAL_SYSTEM,
        messages   = [{"role": "user", "content": user_msg}],
    )
    raw = resp.content[0].text

    # Parse JSON from response
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found in judge response: {raw[:200]}")
    data = json.loads(match.group())

    # Compute overall_score as mean if not explicitly returned
    scores = [data["reasoning_quality"]["score"],
              data["intellectual_depth"]["score"],
              data["reasoning_traceability"]["score"]]
    data["overall_score"] = round(sum(scores) / 3, 2)
    return EvalResult(**data)


## 4. Run Evaluation — All Models

Iterate over each model's inference results and score each of the 100 responses.

In [ ]:
def run_evaluation(model_name: str, data_file: Path, out_file: Path,
                   reasoning_col: str = "thinking"):
    """Score all responses in data_file and save results to out_file.

    Args:
        model_name:    Human-readable model label (e.g. 'Graph-PRefLexOR-8B').
        data_file:     JSONL file with inference outputs (one row per question).
        out_file:      JSONL file where evaluation scores are written.
        reasoning_col: Column containing the reasoning trace (use 'raw_output' for
                       Graph-PRefLexOR, 'thinking' for Qwen, or '' for Llama).
    """
    df = pd.read_json(data_file, lines=True)
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=model_name):
        reasoning = str(row.get(reasoning_col) or row.get("raw_output") or "").strip()
        try:
            result = evaluate(
                model_name      = model_name,
                question        = row["question"],
                correct_answer  = row.get("accepted_answer", ""),
                model_reasoning = reasoning,
            )
            results.append({
                "id":                       row.get("id"),
                "model":                    model_name,
                "reasoning_quality_score":  result.reasoning_quality.score,
                "intellectual_depth_score": result.intellectual_depth.score,
                "reasoning_traceability_score": result.reasoning_traceability.score,
                "overall_score":            result.overall_score,
            })
        except Exception as e:
            print(f"  [error] row {row.get('id')}: {e}")
            results.append({"id": row.get("id"), "model": model_name,
                             "reasoning_quality_score": 0, "intellectual_depth_score": 0,
                             "reasoning_traceability_score": 0, "overall_score": 0})
        time.sleep(0.2)   # respect API rate limits

    pd.DataFrame(results).to_json(out_file, orient="records", lines=True)
    print(f"Saved {len(results)} scores → {out_file}")


# ── Uncomment each block to run evaluation for that model ──────────────────

# Graph-PRefLexOR-8B  (Qwen3-8B base, GRPO fine-tuned)
# run_evaluation("Graph-PRefLexOR-8B",
#                DATA_DIR / "graph_8b_data_eval_all.jsonl",
#                DATA_DIR / "graph_8b_res_eval.jsonl",
#                reasoning_col="raw_output")

# Qwen3-8B (thinking enabled)
# run_evaluation("Qwen3-8B",
#                DATA_DIR / "qwen_8b_data_eval_all.jsonl",
#                DATA_DIR / "qwen_8b_res_eval.jsonl",
#                reasoning_col="thinking")

# Qwen3-8B (no-thinking, ablation)
# run_evaluation("Qwen3-8B-NoThinking",
#                DATA_DIR / "qwen_8b_no_thinking_data_eval_all.jsonl",
#                DATA_DIR / "qwen_8b_no_thinking_res_eval.jsonl",
#                reasoning_col="raw_output")

# Graph-PRefLexOR-3B  (Llama-3.2-3B base, GRPO fine-tuned)
# run_evaluation("Graph-PRefLexOR-3B",
#                DATA_DIR / "graph_3b_data_eval_all.jsonl",
#                DATA_DIR / "graph_3b_res_eval.jsonl",
#                reasoning_col="raw_output")

# Llama-3.2-3B-Instruct (no thinking phase — evaluate final answer directly)
# run_evaluation("Llama-3.2-3B-Instruct",
#                DATA_DIR / "llama_3b_data_eval_all.jsonl",
#                DATA_DIR / "llama_3b_res_eval.jsonl",
#                reasoning_col="")

# Graph-PRefLexOR-1.7B (Qwen3-1.7B base, GRPO fine-tuned)
# run_evaluation("Graph-PRefLexOR-1.7B",
#                DATA_DIR / "graph_1_7b_data_eval_all.jsonl",
#                DATA_DIR / "graph_1_7b_res_eval.jsonl",
#                reasoning_col="raw_output")

# Qwen3-1.7B (thinking enabled)
# run_evaluation("Qwen3-1.7B",
#                DATA_DIR / "qwen_1_7b_data_eval_all.jsonl",
#                DATA_DIR / "qwen_1_7b_res_eval.jsonl",
#                reasoning_col="thinking")

# Qwen3-1.7B (no-thinking, ablation)
# run_evaluation("Qwen3-1.7B-NoThinking",
#                DATA_DIR / "qwen_1_7b_data_nt_eval_all.jsonl",
#                DATA_DIR / "qwen_1_7b_no_thinking_res_eval.jsonl",
#                reasoning_col="raw_output")


## 5. Load Results and Compute Summary Statistics

In [ ]:
def load_eval_results(model_name: str, res_file: Path) -> pd.DataFrame:
    """Load evaluation scores for one model and add a 'model' column."""
    df = pd.read_json(res_file, lines=True)
    df["model"] = model_name
    return df

# Load pre-computed evaluation results for all models
model_files = {
    "Graph-PRefLexOR-8B":   "graph_8b_res_eval.jsonl",
    "Qwen3-8B":             "qwen_8b_res_eval.jsonl",
    "Qwen3-8B-NoThinking":  "qwen_8b_no_thinking_res_eval.jsonl",  # ablation (Section 2.1)
    "Graph-PRefLexOR-3B":   "graph_3b_res_eval.jsonl",
    "Llama-3.2-3B-Instruct":"llama_3b_res_eval.jsonl",
    "Graph-PRefLexOR-1.7B": "graph_1_7b_res_eval.jsonl",
    "Qwen3-1.7B":           "qwen_1_7b_res_eval.jsonl",
    "Qwen3-1.7B-NoThinking":"qwen_1_7b_no_thinking_res_eval.jsonl",  # ablation
}

dfs = []
for model_name, fname in model_files.items():
    fpath = DATA_DIR / fname
    if fpath.exists():
        dfs.append(load_eval_results(model_name, fpath))
    else:
        print(f"  [missing] {fpath.name}")

all_results = pd.concat(dfs, ignore_index=True)

# Compute mean scores per model
metrics = ["reasoning_quality_score", "intellectual_depth_score",
           "reasoning_traceability_score", "overall_score"]
summary = all_results.groupby("model")[metrics].mean().round(2)
print(summary.to_string())


## 6. Reproduce Figure 1 — Bar Chart Comparison

Grouped bar charts comparing Graph-PRefLexOR against base models at each scale. Corresponds to Figure 1 (a–d) in the paper.

In [ ]:
def plot_reasoning_comparison(graph_model: str, base_models: list,
                               title: str, save_path: Path = None):
    """Plot a grouped bar chart comparing one Graph-PRefLexOR model against its baselines.

    Args:
        graph_model: Name of the Graph-PRefLexOR model (must be in summary index).
        base_models: List of baseline model names to compare against.
        title:       Chart title.
        save_path:   If provided, save the figure to this path.
    """
    metric_labels = {
        "reasoning_quality_score":      "Reasoning\nQuality",
        "intellectual_depth_score":     "Intellectual\nDepth",
        "reasoning_traceability_score": "Reasoning\nTraceability",
        "overall_score":                "Overall",
    }
    x       = np.arange(len(metric_labels))
    width   = 0.18
    fig, ax = plt.subplots(figsize=(9, 5), dpi=150)

    all_models = [graph_model] + base_models
    colors     = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

    for i, model in enumerate(all_models):
        if model not in summary.index:
            print(f"  [skip] {model} not in results")
            continue
        scores = [summary.loc[model, m] for m in metric_labels]
        bars   = ax.bar(x + (i - len(all_models)/2 + 0.5) * width, scores,
                        width * 0.9, label=model, color=colors[i % len(colors)])
        # Annotate bars with numeric values
        for bar, score in zip(bars, scores):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                    f"{score:.2f}", ha="center", va="bottom", fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels.values(), fontsize=10)
    ax.set_ylabel("Score (0–10)", fontsize=11)
    ax.set_ylim(0, 10)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=8, loc="upper left")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"Saved → {save_path}")
    plt.show()


# Figure 1 (a) — 8B scale
plot_reasoning_comparison(
    graph_model  = "Graph-PRefLexOR-8B",
    base_models  = ["Qwen3-8B", "Qwen3-8B-NoThinking"],
    title        = "Reasoning Trace Evaluation: 8B Models",
    save_path    = FIGURES_DIR / "reasoning_comparison_8b.png",
)

# Figure 1 (b) — 3B scale
plot_reasoning_comparison(
    graph_model = "Graph-PRefLexOR-3B",
    base_models = ["Llama-3.2-3B-Instruct"],
    title       = "Reasoning Trace Evaluation: 3B Models",
    save_path   = FIGURES_DIR / "reasoning_comparison_3b.png",
)

# Figure 1 (c) — 1.7B scale
plot_reasoning_comparison(
    graph_model = "Graph-PRefLexOR-1.7B",
    base_models = ["Qwen3-1.7B", "Qwen3-1.7B-NoThinking"],
    title       = "Reasoning Trace Evaluation: 1.7B Models",
    save_path   = FIGURES_DIR / "reasoning_comparison_1_7b.png",
)

# Figure 1 (d) — Cross-scale Graph-PRefLexOR comparison
plot_reasoning_comparison(
    graph_model = "Graph-PRefLexOR-8B",
    base_models = ["Graph-PRefLexOR-3B", "Graph-PRefLexOR-1.7B"],
    title       = "Reasoning Trace Evaluation: Graph-PRefLexOR Across Scales",
    save_path   = FIGURES_DIR / "reasoning_comparison_scale.png",
)
